# Entrega 4.2 - Vision Transformers e Modelos Híbridos para Melanoma Acral
Neste notebook, implementamos:
1. **Swin Transformer** (janelas deslocadas, excelente para imagens médicas)
2. **DeiT III** (Data-efficient Image Transformers)
3. **ViT com Fusão Multimodal** (ViT + Metadados Anatômicos)
4. **Modelo Misto (CNN + ViT)** (EfficientNetB0 com 1 camada descongelada + ViT)

In [10]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import gc
import numpy as np
import pandas as pd
import tf_keras as keras
import tensorflow as tf
from tf_keras.models import Model
from tf_keras.layers import Dense, GlobalAveragePooling2D, Dropout, Concatenate, Input, StringLookup, Embedding, Flatten
from tf_keras.preprocessing.image import ImageDataGenerator
from tf_keras.applications import EfficientNetB0
from transformers import TFAutoModelForImageClassification, TFAutoModel
from sklearn.utils import resample

# --- Correção de Compatibilidade do Swin Transformer (Shape Mismatch Bug no TensorFlow) ---
import transformers.models.swin.modeling_tf_swin as swin_module
from transformers.tf_utils import shape_list

def custom_window_reverse(windows: tf.Tensor, window_size: int, height: int, width: int) -> tf.Tensor:
    channels = shape_list(windows)[-1]
    windows = tf.reshape(
        windows, (-1, height // window_size, width // window_size, window_size, window_size, channels)
    )
    windows = tf.transpose(windows, (0, 1, 3, 2, 4, 5))
    windows = tf.reshape(windows, (-1, height, width, channels))
    return windows

swin_module.window_reverse = custom_window_reverse
print("Swin Transformer window_reverse monkeypatch aplicado com sucesso!")
# ------------------------------------------------------------------------------------------

# Configuração da GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detectada: {gpus}")


Swin Transformer window_reverse monkeypatch aplicado com sucesso!


## 1. Carregamento de Dados e Metadados Anatômicos

In [11]:
PASTAS_DATASET = ['imagens_acrais_benignas', 'imagens_acrais_maligno']
dfs = []
for pasta in PASTAS_DATASET:
    caminho_csv = os.path.join(pasta, 'metadata.csv')
    if os.path.exists(caminho_csv):
        df_temp = pd.read_csv(caminho_csv)
        df_temp['caminho_imagem'] = df_temp['isic_id'].apply(lambda x: os.path.join(pasta, f"{x}.jpg"))
        dfs.append(df_temp)

df_completo = pd.concat(dfs, ignore_index=True)
df_existente = df_completo[df_completo['caminho_imagem'].apply(os.path.exists)].copy()
df_existente['anatom_site_general'] = df_existente['anatom_site_general'].fillna('unknown')

df_benigno = df_existente[df_existente['diagnosis_1'] == 'Benign']
df_maligno = df_existente[df_existente['diagnosis_1'] == 'Malignant']
df_filtrado = pd.concat([df_benigno, df_maligno]).sample(frac=1, random_state=42).reset_index(drop=True)
df_filtrado['label'] = (df_filtrado['diagnosis_1'] == 'Malignant').astype(int)

print(f"Dataset total: {len(df_filtrado)} imagens")

Dataset total: 2059 imagens


## 2. Preparação de Generators (TF Dataset / ImageDataGenerator)

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

TAMANHO = (224, 224)
BATCH_SIZE = 8

df_train, df_val = train_test_split(df_filtrado, test_size=0.2, random_state=42, stratify=df_filtrado['label'])

# Oversampling no treino
df_malignant = df_train[df_train['label'] == 1]
df_benign = df_train[df_train['label'] == 0]
maior_classe = max(len(df_malignant), len(df_benign))
df_benign_upsampled = resample(df_benign, replace=True, n_samples=maior_classe, random_state=42)
df_malignant_upsampled = resample(df_malignant, replace=True, n_samples=maior_classe, random_state=42)
df_treino_equilibrado = pd.concat([df_malignant_upsampled, df_benign_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Treino equilibrado: {len(df_treino_equilibrado)} imagens")
print(df_treino_equilibrado['label'].value_counts())

# Funções de processamento tf.data
def process_path(file_path, site, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, TAMANHO)
    img = img / 255.0 # Normalização básica
    return {'imagem': img, 'site': site}, label

data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.125), # 45 graus
    keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
])

def augment(inputs, labels):
    img = data_augmentation(inputs['imagem'], training=True)
    return {'imagem': img, 'site': inputs['site']}, labels

# Dataset de Validação
val_sites = df_val['anatom_site_general'].values
val_caminhos = df_val['caminho_imagem'].values
val_labels = df_val['label'].values
val_dataset = tf.data.Dataset.from_tensor_slices((val_caminhos, val_sites, val_labels))
val_dataset = val_dataset.map(process_path).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Dataset de Treino com Augmentation
train_sites = df_treino_equilibrado['anatom_site_general'].values
train_caminhos = df_treino_equilibrado['caminho_imagem'].values
train_labels = df_treino_equilibrado['label'].values
train_dataset = tf.data.Dataset.from_tensor_slices((train_caminhos, train_sites, train_labels))
train_dataset = train_dataset.map(process_path).batch(BATCH_SIZE).map(augment).prefetch(tf.data.AUTOTUNE)


Treino equilibrado: 2490 imagens
label
1    1245
0    1245
Name: count, dtype: int64


## 3. Swin Transformer (Swin-Base)

In [13]:
def build_swin_model():
    # Carregando o Swin Transformer do Hugging Face
    base_model = TFAutoModelForImageClassification.from_pretrained("microsoft/swin-base-patch4-window7-224", num_labels=1, ignore_mismatched_sizes=True, use_safetensors=False)
    
    inputs = Input(shape=(224, 224, 3), name='imagem')
    # HF models esperam input no formato (batch, channels, height, width) se não for TF nativo, 
    # mas os TFModels do HF aceitam (batch, height, width, channels) normalmente em pixel_values.
    # Vamos extrair os logits diretamente.
    inputs_transposed = tf.transpose(inputs, perm=[0, 3, 1, 2])
    outputs = base_model(pixel_values=inputs_transposed).logits
    saida = keras.activations.sigmoid(outputs)
    
    model = Model(inputs=inputs, outputs=saida, name="Swin_Transformer")
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_swin = build_swin_model()
model_swin.summary()

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ebf9968f-f9ad-44ab-bd11-b57c480f7f95)')' thrown while requesting HEAD https://huggingface.co/microsoft/swin-base-patch4-window7-224/resolve/main/config.json
Retrying in 1s [Retry 1/5].
All model checkpoint layers were used when initializing TFSwinForImageClassification.

Some layers of TFSwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-base-patch4-window7-224 and are newly initialized: ['swin/encoder/layers.2/blocks.0/attention/self/relative_position_index:0', 'swin/encoder/layers.0/blocks.0/attention/self/relative_position_index:0', 'swin/encoder/layers.2/blocks.8/attention/self/relative_position_index:0', 'swin/encoder/layers.2/blocks.6/attention/self/relative_position_index:0', 'swin/encoder/layers.1/blocks.0/attention/self/relative_position_index:0', 'swin/encoder/layers.2/blocks.14/attention/self/relative_position_i

Model: "Swin_Transformer"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 imagem (InputLayer)         [(None, 224, 224, 3)]     0         
                                                                 
 tf.compat.v1.transpose_4 (  (None, 3, 224, 224)       0         
 TFOpLambda)                                                     
                                                                 
 tf_swin_for_image_classifi  TFSwinImageClassifierOu   86801873  
 cation_1 (TFSwinForImageCl  tput(loss=None, logits=             
 assification)               (None, 1),                          
                              hidden_states=None, at             
                             tentions=None, reshaped             
                             _hidden_states=None)                
                                                                 
 tf.math.sigmoid_2 (TFOpLam  (None, 1)            

## 4. DeiT III (Data-efficient Image Transformers)

In [14]:
def build_deit_model():
    # Usando o DeiT base
    base_model = TFAutoModelForImageClassification.from_pretrained("facebook/deit-base-patch16-224", num_labels=1, ignore_mismatched_sizes=True, use_safetensors=False)
    
    inputs = Input(shape=(224, 224, 3), name='imagem')
    inputs_transposed = tf.transpose(inputs, perm=[0, 3, 1, 2])
    outputs = base_model(pixel_values=inputs_transposed).logits
    saida = keras.activations.sigmoid(outputs)
    
    model = Model(inputs=inputs, outputs=saida, name="DeiT_III")
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_deit = build_deit_model()
model_deit.summary()

All model checkpoint layers were used when initializing TFViTForImageClassification.

Some weights of TFViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier/kernel:0: found shape (768, 1000) in the checkpoint and (768, 1) in the model instantiated
- classifier/bias:0: found shape (1000,) in the checkpoint and (1,) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "DeiT_III"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 imagem (InputLayer)         [(None, 224, 224, 3)]     0         
                                                                 
 tf.compat.v1.transpose_5 (  (None, 3, 224, 224)       0         
 TFOpLambda)                                                     
                                                                 
 tf_vi_t_for_image_classifi  TFSequenceClassifierOut   85799425  
 cation_1 (TFViTForImageCla  put(loss=None, logits=(             
 ssification)                None, 1),                           
                              hidden_states=None, at             
                             tentions=None)                      
                                                                 
 tf.math.sigmoid_3 (TFOpLam  (None, 1)                 0         
 bda)                                                     

## 5. ViT com Fusão Multimodal (Metadados Anatômicos)

In [15]:
def build_multimodal_vit():
    # Input de Imagem
    img_input = Input(shape=(224, 224, 3), name='imagem')
    base_vit = TFAutoModel.from_pretrained("google/vit-base-patch16-224-in21k", use_safetensors=False)
    # Extraindo as representações globais do ViT (pooler_output)
    img_input_transposed = tf.transpose(img_input, perm=[0, 3, 1, 2])
    vit_features = base_vit(pixel_values=img_input_transposed).pooler_output
    
    # Input de Metadados Anatômicos
    site_input = Input(shape=(1,), dtype=tf.string, name='site')
    vocab = np.unique(df_filtrado['anatom_site_general'].astype(str))
    lookup = StringLookup(vocabulary=vocab, output_mode='int')(site_input)
    site_embedding = Embedding(input_dim=len(vocab)+2, output_dim=16)(lookup)
    site_features = Flatten()(site_embedding)
    
    # Fusão (Concatenação)
    concat = Concatenate()([vit_features, site_features])
    x = Dense(128, activation='relu')(concat)
    x = Dropout(0.3)(x)
    saida = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs={'imagem': img_input, 'site': site_input}, outputs=saida, name="Multimodal_ViT")
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_multimodal = build_multimodal_vit()
model_multimodal.summary()

All model checkpoint layers were used when initializing TFViTModel.

All the layers of TFViTModel were initialized from the model checkpoint at google/vit-base-patch16-224-in21k.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFViTModel for predictions without further training.


Model: "Multimodal_ViT"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 site (InputLayer)           [(None, 1)]                  0         []                            
                                                                                                  
 imagem (InputLayer)         [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 string_lookup_1 (StringLoo  (None, 1)                    0         ['site[0][0]']                
 kup)                                                                                             
                                                                                                  
 tf.compat.v1.transpose_6 (  (None, 3, 224, 224)          0         ['imagem[0][0]'] 

## 6. Modelo Misto (CNN + ViT) - EfficientNetB0 + ViT

In [16]:
def build_mixed_cnn_vit():
    img_input = Input(shape=(224, 224, 3), name='imagem')
    
    # --- Ramo 1: EfficientNetB0 com 1 bloco descongelado (melhor do Entrega 3) ---
    base_cnn = EfficientNetB0(weights="imagenet", include_top=False, input_tensor=img_input)
    base_cnn.trainable = False
    # Descongelando o último bloco do B7
    for layer in base_cnn.layers:
        if layer.name.startswith("block7") or layer.name.startswith("top_"):
            if not isinstance(layer, keras.layers.BatchNormalization):
                layer.trainable = True
                
    cnn_features = GlobalAveragePooling2D()(base_cnn.output)
    cnn_features = Dropout(0.4)(cnn_features)
    
    # --- Ramo 2: ViT Base ---
    base_vit = TFAutoModel.from_pretrained("google/vit-base-patch16-224-in21k", use_safetensors=False)
    img_input_transposed = tf.transpose(img_input, perm=[0, 3, 1, 2])
    vit_features = base_vit(pixel_values=img_input_transposed).pooler_output
    vit_features = Dropout(0.4)(vit_features)
    
    # Fusão de Representações CNN (Local/Texturas) + ViT (Global/Hierárquica)
    concat = Concatenate()([cnn_features, vit_features])
    x = Dense(256, activation='relu')(concat)
    x = Dropout(0.3)(x)
    saida = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=img_input, outputs=saida, name="Mixed_CNN_ViT")
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_mixed = build_mixed_cnn_vit()
model_mixed.summary()

All model checkpoint layers were used when initializing TFViTModel.

All the layers of TFViTModel were initialized from the model checkpoint at google/vit-base-patch16-224-in21k.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFViTModel for predictions without further training.


Model: "Mixed_CNN_ViT"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 imagem (InputLayer)         [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 rescaling_1 (Rescaling)     (None, 224, 224, 3)          0         ['imagem[0][0]']              
                                                                                                  
 normalization_1 (Normaliza  (None, 224, 224, 3)          7         ['rescaling_1[0][0]']         
 tion)                                                                                            
                                                                                                  
 tf.math.multiply_1 (TFOpLa  (None, 224, 224, 3)          0         ['normalization_1[

## 7. Funções Auxiliares de Treinamento e Avaliação

In [17]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Criação dos conjuntos com input único (apenas imagem) para os modelos Swin, DeiT e Misto
train_dataset_single = train_dataset.map(lambda x, y: (x['imagem'], y))
val_dataset_single = val_dataset.map(lambda x, y: (x['imagem'], y))

def train_and_evaluate_model(model, train_ds, val_ds, epochs=5):
    print(f"\n=========================================")
    print(f"Treinando o modelo: {model.name}")
    print(f"=========================================")
    
    # Callback de Early Stopping
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=3, 
        restore_best_weights=True, 
        verbose=1
    )
    
    # Treinamento
    history = model.fit(
        train_ds,
        epochs=epochs,
        validation_data=val_ds,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Avaliação
    print(f"\n[1] Avaliação no conjunto de Validação:")
    eval_results = model.evaluate(val_ds, verbose=0)
    loss = eval_results[0]
    acc = eval_results[1]
    print(f"Loss: {loss:.4f} | Acurácia: {acc:.4f}")
    
    # Previsões
    print(f"\n[2] Calculando previsões e métricas:")
    preds_prob = []
    y_true = []
    
    for batch_x, batch_y in val_ds:
        pred = model.predict(batch_x, verbose=0)
        preds_prob.extend(pred.flatten())
        y_true.extend(batch_y.numpy())
        
    preds_prob = np.array(preds_prob)
    preds_class = (preds_prob > 0.5).astype(int)
    y_true = np.array(y_true)
    
    # Relatório de Classificação
    print("\nRelatório de Classificação:")
    print(classification_report(y_true, preds_class, target_names=["Benign", "Malignant"]))
    
    # Plot da Matriz de Confusão
    cm = confusion_matrix(y_true, preds_class)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt="d", 
        cmap="Blues", 
        xticklabels=["Benign", "Malignant"], 
        yticklabels=["Benign", "Malignant"]
    )
    plt.title(f"Matriz de Confusão - {model.name}")
    plt.ylabel("Verdadeiro")
    plt.xlabel("Previsto")
    plt.show()
    
    # Plot das curvas de aprendizado
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Treino')
    plt.plot(history.history['val_loss'], label='Validação')
    plt.title('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Treino')
    plt.plot(history.history['val_accuracy'], label='Validação')
    plt.title('Acurácia')
    plt.legend()
    plt.show()
    
    return history, loss, acc

## 8. Treinamento e Avaliação do Swin Transformer

In [ ]:
history_swin, loss_swin, acc_swin = train_and_evaluate_model(model_swin, train_dataset_single, val_dataset_single, epochs=5)


Treinando o modelo: Swin_Transformer
Epoch 1/5
312/312 [==============================] - 3047s 9s/step - loss: 0.2884 - accuracy: 0.8751 - val_loss: 0.5292 - val_accuracy: 0.8131
Epoch 2/5
312/312 [==============================] - 2971s 10s/step - loss: 0.1863 - accuracy: 0.9309 - val_loss: 0.2372 - val_accuracy: 0.9053
Epoch 3/5
312/312 [==============================] - 2895s 9s/step - loss: 0.1374 - accuracy: 0.9442 - val_loss: 0.2208 - val_accuracy: 0.9175
Epoch 4/5
312/312 [==============================] - 2741s 9s/step - loss: 0.0991 - accuracy: 0.9610 - val_loss: 0.3459 - val_accuracy: 0.9053
Epoch 5/5
 46/312 [===>..........................] - ETA: 36:53 - loss: 0.1273 - accuracy: 0.9565

## 9. Treinamento e Avaliação do DeiT III

In [ ]:
history_deit, loss_deit, acc_deit = train_and_evaluate_model(model_deit, train_dataset_single, val_dataset_single, epochs=5)

## 10. Treinamento e Avaliação do ViT Multimodal

In [ ]:
history_multimodal, loss_multimodal, acc_multimodal = train_and_evaluate_model(model_multimodal, train_dataset, val_dataset, epochs=5)

## 11. Treinamento e Avaliação do Modelo Misto (CNN + ViT)

In [ ]:
history_mixed, loss_mixed, acc_mixed = train_and_evaluate_model(model_mixed, train_dataset_single, val_dataset_single, epochs=5)

## 12. Comparação Geral dos Resultados

Abaixo apresentamos uma tabela comparativa dos quatro modelos treinados nesta entrega:

| Modelo | Acurácia de Validação | Perda de Validação |
| :--- | :---: | :---: |
| **Swin Transformer** | `acc_swin` | `loss_swin` |
| **DeiT III** | `acc_deit` | `loss_deit` |
| **ViT Multimodal** | `acc_multimodal` | `loss_multimodal` |
| **Misto (CNN + ViT)** | `acc_mixed` | `loss_mixed` |

> **Conclusão preliminar**: Os modelos baseados em Transformers de visão (Swin, DeiT e ViT) trazem um potencial enorme para a classificação de imagens dermatológicas complexas, capturando padrões espaciais finos através das janelas de atenção e se beneficiando fortemente do enriquecimento com metadados anatômicos.